文档加载器 Document Loaders

数据源可能包含多种格式的文件，如文本文档、Markdown，PDF 等。LangChain 实现和集成了众多文档加载器（https://docs.langchain.com/oss/python/integrations/document_loaders ），方便从不同格式的文件中加载数据。

常用Loaders：
- TextLoader：文本文件。
- CSVLoader：CSV文件。
- PyPDFLoader：PDF文件。
- WebBaseLoader：网页。

LangChain的设计：对于 Source 中多种不同的数据源，我们可以用一种统一的形式读取、调用。上述每一个文档加载器，都要继承自 `BaseLoader` 基类，此类提供了通用的 load （一次加载所有文档）与 lazy_load （以延迟方式加载文档） 方法，用于从数据源加载数据并处理为 Document 对象 。

## 1、加载txt

In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

text_loader = TextLoader(file_path="../asset/load/01-langchain-utf-8.txt", encoding="utf-8")

docs = text_loader.load()

print(docs)

[Document(metadata={'source': '../asset/load/01-langchain-utf-8.txt'}, page_content='LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务')]


Document对象中有两个重要的属性：
- page_content：真正的文档内容，字符串类型。
- metadata：文档内容的原数据，字典类型。

In [3]:
print(type(docs[0]))
print(docs[0].metadata)
print(docs[0].page_content)

<class 'langchain_core.documents.base.Document'>
{'source': '../asset/load/01-langchain-utf-8.txt'}
LangChain 是一个用于构建基于大语言模型（LLM）应用的开发框架，旨在帮助开发者更高效地集成、管理和增强大语言模型的能力，构建端到端的应用程序。它提供了一套模块化工具和接口，支持从简单的文本生成到复杂的多步骤推理任务


## 2、加载CSV

举例：加载CSV所有列

In [6]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path="../asset/load/02-load.csv")
data = loader.load()
print(data)

print(type(data))
print(type(data[0]))
print(len(data))
print(data[0].metadata)
print(data[0].page_content)

[Document(metadata={'source': '../asset/load/02-load.csv', 'row': 0}, page_content='id: 1\ntitle: Introduction to Python\ncontent: Python is a popular programming language.\nauthor: John Doe'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 1}, page_content='id: 2\ntitle: Data Science Basics\ncontent: Data science involves statistics and machine learning.\nauthor: Jane Smith'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 2}, page_content='id: 3\ntitle: Web Development\ncontent: HTML, CSS and JavaScript are core web technologies.\nauthor: Mike Johnson'), Document(metadata={'source': '../asset/load/02-load.csv', 'row': 3}, page_content='id: 4\ntitle: Artificial Intelligence\ncontent: AI is transforming many industries.\nauthor: Sarah Williams')]
<class 'list'>
<class 'langchain_core.documents.base.Document'>
4
{'source': '../asset/load/02-load.csv', 'row': 0}
id: 1
title: Introduction to Python
content: Python is a popular programming language.
auth

## 3、加载JSON

LangChain提供的JSON格式的文档加载器是JSONLoader。在实际应用场景中，JSON格式的数据占很大比例，而且JSON的形式也是多样的，需要特别关注。

JSONLoader使用指定的jq结构来解析JSON文件。jq是一个轻量级的命令行JSON处理器，可以对JSON格式的数据进行各种复杂的处理，包括数据过滤、映射、减少和转换，是处理JSON数据的首选工具之一。
```
pip install jq
```
常见 jq schema 参考：
```
JSON -> ["...", "...", "..."]
jq_schema -> ".[]"

JSON -> [{"text": ...}, {"text": ...}, {"text": ...}]
jq_schema -> ".[].text"

JSON -> {"key": [{"text": ...}, {"text": ...}, {"text": ...}]}
jq_schema -> ".key[].text"
```
详细用法可参考 https://jqlang.org/manual/#basic-filters。

举例1：使用JSONLoader文档加载器加载

In [9]:
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# json_loader = JSONLoader(
#     file_path="../asset/load/03-load.json",
#     jq_schema=".", # 提取所有字段
#     text_content=False # 保持原始 JSON 结构，将提取的数据转换为JSON字符串存入page_content字段中
# )

# 遍历指定元素
json_loader = JSONLoader(
    file_path="../asset/load/03-load.json",
    jq_schema=".messages[].content"  # 遍历.messages[]中所有元素 从每一个元素中提取.content字段
)

docs = json_loader.load()
rprint(docs)


[
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 1
        },
        page_content='Hello, how are you today?'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 2
        },
        page_content="I'm doing well, thanks for asking!"
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 3
        },
        page_content='Would you like to meet for lunch?'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-load.json',
            'seq_num': 4
        },
        page_content='Sure, that sounds great!'
    )
]

举例2：提取03-response.json文件中指定的文本

In [10]:
# 1.导入相关依赖
from langchain_community.document_loaders import JSONLoader
from rich import print as rprint

# 2.定义json文件的路径
file_path = '../asset/load/03-response.json'

# 3.定义JSONLoader对象
# 需求1：提取data.items中的数据
loader = JSONLoader(
    file_path=file_path, # 文件路径
    jq_schema=".data.items[]",
    text_content=False, # 提取内容是否为字符串格式
)

# 需求2：提取data.items[].content中的数据
loader = JSONLoader(
    file_path=file_path, # 文件路径
    jq_schema=".data.items[].content",
)

loader = JSONLoader(
    file_path=file_path,
    jq_schema="""
        .data.items[] | {
            author,
            created_at,
            content: (.title + "\n" + .content)
        }
    """,
    text_content=False
)

data = loader.load()
rprint(data)

[
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 1
        },
        page_content='{"author": {"id": "user_1", "name": "Alice"}, "created_at": "2023-10-05T08:12:33Z", 
"content": "Understanding JSONLoader\\nThis article explains how to parse API responses..."}'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 2
        },
        page_content='{"author": {"id": "user_2", "name": "Bob"}, "created_at": "2023-10-05T09:15:21Z", "content": 
"Advanced jq Schema Patterns\\nLearn to handle nested structures with..."}'
    ),
    Document(
        metadata={
            'source': 
'/Users/lxp/Documents/学习笔记/learnNote/AI/LangChain/langchain1.2-tutorial/asset/load/03-response.json',
            'seq_num': 3
        },
        page_content='{"author": {"id": "user_3", "name": "Charlie"}, "created_at": "2023-10-05T10:03:47Z", 
"content": "LangChain Metadata Handling\\nBest practices for preserving metadata..."}'
    )
]

## 4、加载pdf

PDF 存在多种来源格式，包括扫描版（图片 PDF）、电子文本版、混合版。并且布局格式也多种多样，包括单列布局、双列布局甚至竖排文本布局。并且包含段落、标题、页眉页脚、表格、数学公式、化学式、特殊符号、图片等各种元素。

因此，PDF 解析存在很多挑战。对于复杂 PDF，需要进行文本提取、布局检测、表格解析、公式识别等处理。

### 4.1 方式1:PyPDFLoader

LangChain加载PDF文件使用的是pypdf。
```
pip install pypdf
```


In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    # 文件路径，支持本地文件和在线文件链接
    #file_path="../asset/load/04-sample.pdf",
    file_path="https://arxiv.org/pdf/alg-geom/9202012",
    # 提取模式:控制如何从 PDF 文件中解析和提取文本结构。
    # plain 提取文本，默认值
    # layout 布局感知提取模式，通常会通过插入大量的空格、换行符，来模拟原文档中的多栏、缩进和间距（适用场景：学术论文（如 arXiv 论文）、多栏报刊杂志、带有左右分栏的合同）
    extraction_mode="plain",
)

docs = loader.load()
print(len(docs))

2


### 4.2 方式2:MinerU

[MinerU](https://mineru.net/) 提供了 PDF、Word、PPT、图片等文件的解析，支持图像提取、OCR、公式、表格解析等功能。

调用在线服务：https://mineru.net/apiManage/docs。 可以从本地批量上传文件进行解析，并接收解析结果。

In [ ]:
import time
import requests
import os
from dotenv import load_dotenv

load_dotenv(override=True)

def upload_files(file_paths: list[str]) -> str:
    token = os.getenv("MINERU_TOKEN")
    url = "https://mineru.net/api/v4/file-urls/batch"
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    files_info = [
        {
            "name": os.path.basename(file_path),
            "is_ocr": True, # 是否启动 ocr 功能
            "data_id": f"file_{i}" # 解析对象对应的数据 ID
        } for i, file_path in enumerate(file_paths)
    ]
    data = {
        "enable_formula": True,  # 是否开启公式识别
        "enable_table": True,  # 是否开启表格识别
        "language": "ch",
        "files": files_info,
    }

    try:
        response = requests.post(url,headers=header,json=data)
        if response.status_code == 200:
            result = response.json()
            print('response success. result:{}'.format(result))
            if result["code"] == 0:
                batch_id = result["data"]["batch_id"]
                urls = result["data"]["file_urls"]
                print('batch_id:{},urls:{}'.format(batch_id, urls))
                for i in range(0, len(urls)):
                    with open(file_paths[i], 'rb') as f:
                        res_upload = requests.put(urls[i], data=f)
                        if res_upload.status_code == 200:
                            print(f"{urls[i]} upload success")
                        else:
                            print(f"{urls[i]} upload failed")
                            return None
                return batch_id
            else:
                print('apply upload url failed,reason:{}'.format(result["msg"]))
                return None
        else:
            print('response not success. status:{} ,result:{}'.format(response.status_code, response))
            return None
    except Exception as err:
        print(err)
        return None

def download_files(batch_id):
    if not batch_id:
        print("batch_id为空，跳过下载")
        return

    os.makedirs("parsed_files", exist_ok=True)
    token = os.getenv("MINERU_TOKEN")
    url = f"https://mineru.net/api/v4/extract-results/batch/{batch_id}"
    header = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {token}"
    }

    failed_files = set()
    done_files = set()

    while True:
        res = requests.get(url, headers=header)
        result_json = res.json()

        if res.status_code != 200 or result_json.get("code") != 0:
            print("get result failed:", result_json)
            break

        extract_results = result_json["data"]["extract_result"]

        for result in extract_results:
            data_id = result["data_id"]
            if result["state"] == "failed":
                failed_files.add(data_id)
            elif result["state"] == "done" and data_id not in done_files:
                done_files.add(data_id)
                full_zip_url = result["full_zip_url"]
                res_download = requests.get(full_zip_url, stream=True)

                with open(f"parsed_files/{result['file_name']}_{result['data_id']}.zip", "wb") as f:
                    for chunk in res_download.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

        if len(failed_files) + len(done_files) == len(extract_results):
            break

        time.sleep(5)

    for i in failed_files:
        print("failed:", i)
    for i in done_files:
        print("done:", i)

file_paths = ["../asset/load/04-sample.pdf"]
batch_id = upload_files(file_paths)
print(batch_id)

if batch_id:
    download_files(batch_id)